# colab_04 — Multi-study integration with scVI

Fourth notebook. The three studies are QC'd and saved as per-study processed h5ads
(`li_processed.h5ad` 381,898 cells / `seaad_processed.h5ad` 272,296 / `haney_processed.h5ad`
108,764). This notebook integrates them into one batch-corrected latent space with **scVI**.

**Strategy (locked 2026-05-21):**
- **ITS — integrate-then-subset.** scVI harmonizes the three studies at the whole-tissue
  level here; astrocyte + microglia subsetting happens *after* integration (colab_05), so
  within-glia structure is decided downstream, not by this object's clustering.
- **`batch_key = study_id`.** Donor- and region-level structure is preserved (NOT corrected) —
  correcting on donor would erase the donor-level APOE signal that eval #2 depends on.
- **3000 HVGs, `seurat_v3`, batch-aware**, computed on raw counts across the concatenated
  object; the 8 niche-critical genes are force-injected into the scVI gene set even if not HVG.
- All three studies use **HGNC symbols**, so concatenation is on the gene-symbol **intersection**
  (`join="inner"`).

**What this notebook deliberately does NOT do (deferred):**
- **Annotation + astro/micro subset + the carried-forward Li/Haney niche-mito diagnostic** → colab_05.
- **Second integration method (scANVI / Harmony) + full scIB comparison** → colab_06. Full scIB
  needs bio-conservation metrics (NMI/ARI/ASW vs cell type), which need labels that don't exist
  until colab_05. colab_04 ships only a **label-free residual-batch screen**.

**Methodological choices baked in (auditable):** `gene_likelihood="nb"` (scVI default `zinb`;
nb fits UMI snRNA where dedup absorbs technical dropout), `n_latent=30` (default 10), `n_layers=2`
(default 1), single fixed seed (the N=3 seed protocol applies to the FT runs, not integration).

**Runtime: requires a GPU runtime (A100).** Integration env (`requirements_integration.txt`,
Py3.12, scvi-tools 1.4.3). The QC notebooks ran CPU-only; scVI needs the GPU.

## 1 — Setup

### 1a — Mount Drive + clone/pull repo + install env

Identical pattern to `colab_01`–`colab_03`: mount Drive, clone-or-pull the repo, pin numpy
first, install the integration env. `requirements_integration.txt` now also pulls `scikit-misc`
(provides `skmisc.loess`, required by the `seurat_v3` HVG flavor in 4a).

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info[:2]}"

# Pin numpy first so pip picks numpy-1.x-compatible wheels (same rationale as colab_01-03).
!pip install numpy==1.26.4
!pip install -r {REPO_PATH}/requirements_integration.txt

## 2 — Environment capture

### 2a — pip freeze + env JSON

Snapshot exact versions to `outputs/software_versions/`. Extends the QC-notebook capture with
explicit **scvi-tools / scanpy / anndata / jax** versions — the integration stack is the most
version-fragile part of the pipeline and these belong in the paper Methods. GPU fields should now
be populated (unlike the CPU-only QC runs).

In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_04"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":   NOTEBOOK_ID,
    "date":          TODAY,
    "python_version": sys.version,
    "platform":      platform.platform(),
    "os_release":    platform.release(),
    "gpu":           _run(["nvidia-smi", "-L"]),
    "nvidia_driver": _run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]),
    "git_commit":    _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "scvi_tools_version": _ver("scvi"),
    "scanpy_version":     _ver("scanpy"),
    "anndata_version":    _ver("anndata"),
    "jax_version":        _ver("jax"),
    "scib_metrics_version": _ver("scib_metrics"),
}
try:
    import torch
    env_snapshot["torch_version"]      = torch.__version__
    env_snapshot["torch_cuda_version"] = torch.version.cuda
    env_snapshot["cuda_available"]     = bool(torch.cuda.is_available())
    env_snapshot["cudnn_version"]      = torch.backends.cudnn.version() if torch.cuda.is_available() else None
except ImportError:
    env_snapshot["torch_version"]  = None
    env_snapshot["cuda_available"] = None
    env_snapshot["cudnn_version"]  = None

ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 3 — Load processed studies + concatenate

### 3a — Load three processed h5ads, harmonize obs, sanity-check raw counts

Load each per-study processed object from Drive. Each must carry the harmonized obs schema the QC
notebooks wrote (`REQUIRED_OBS`); fail loud if any column is missing. Verify `.X` is raw counts
(scVI and `seurat_v3` HVG both require counts, not normalized values). Subset obs to the common
schema so the concat stays clean, and preserve SEA-AD's cell-type column as `orig_celltype` (NaN
for the other two) — it is the only inherited annotation and is used as a cross-check in colab_05,
not as a label here. `study_id` is assigned by the concat in 3b, not here.

In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:24s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

PROCESSED_DIR = os.path.join(DRIVE_ROOT, "processed")
STUDY_FILES = {
    "Li2025":    "li_processed.h5ad",
    "SEA-AD":    "seaad_processed.h5ad",
    "Haney2024": "haney_processed.h5ad",
}

# Single-region studies whose QC notebook didn't stamp a constant `region` column
# (Li 2025 = temporal cortex only). Multi-region studies (SEA-AD, Haney) already
# tag per-cell `region` in their QC. Backfilled below before the required-col check
# so the obs schema is uniform; any *other* study missing region still fails loud.
SINGLE_REGION = {"Li2025": "temporal cortex"}

REQUIRED_OBS = ["donor_id", "sample_id", "region", "apoe_genotype", "apoe_carrier",
                "diagnosis", "sex", "n_genes_by_counts", "total_counts",
                "total_counts_mt", "pct_counts_mt"]
# Only SEA-AD ships cell-type labels; preserved for colab_05 cross-check, NaN elsewhere.
CELLTYPE_CANDIDATES = ["Subclass", "subclass", "cell_type", "celltype", "CellType",
                       "supertype", "Supertype", "class", "Class"]

adatas = {}
for study, fn in STUDY_FILES.items():
    path = os.path.join(PROCESSED_DIR, fn)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{study}: missing processed file {path}")
    a = sc.read_h5ad(path)

    if "region" not in a.obs.columns and study in SINGLE_REGION:
        a.obs["region"] = SINGLE_REGION[study]

    missing = [c for c in REQUIRED_OBS if c not in a.obs.columns]
    if missing:
        raise KeyError(f"{study}: obs missing required columns {missing}; "
                       f"have {list(a.obs.columns)}")

    # raw-counts guard: random sample 2000 cells (not head-slice — sorted deposits hide
    # non-integer tail rows), require >=99% integer-valued .X
    _rng = np.random.default_rng(0)
    _idx = _rng.choice(a.n_obs, size=min(2000, a.n_obs), replace=False)
    Xs = a.X[_idx]
    data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
    frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
    if frac_int < 0.99:
        raise ValueError(f"{study}: .X is not raw counts (int frac {frac_int:.3f}); "
                         "scVI + seurat_v3 HVG require raw counts")

    keep = a.obs[REQUIRED_OBS].copy()
    ct_col = next((c for c in CELLTYPE_CANDIDATES if c in a.obs.columns), None)
    keep["orig_celltype"] = a.obs[ct_col].astype(str).values if ct_col else np.nan
    a.obs = keep
    a.var = a.var[[]].copy()          # drop var extras; var_names = HGNC symbols
    adatas[study] = a
    print(f"{study:10s}: {a.shape}  raw-counts int-frac={frac_int:.3f}  cell-type col={ct_col}")
    _ram(f"loaded {study}")

### 3b — Concatenate on the gene-symbol intersection (inner join, sparse)

`inner` join keeps only genes shared by all three studies — the correct choice for integration
(an `outer` join would inject all-zero columns for studies missing a gene, biasing both HVG
selection and scVI). Report per-study and intersection gene counts, fail loud if the intersection
is implausibly small (would signal a symbol-convention mismatch), and report whether each
niche-critical gene survives the intersection. `study_id` is created here from the concat keys.

In [ ]:
gene_sets = {s: set(a.var_names) for s, a in adatas.items()}
inter = set.intersection(*gene_sets.values())
print("per-study gene counts:", {s: len(g) for s, g in gene_sets.items()})
print(f"gene intersection (inner join): {len(inter)}")

MIN_INTERSECTION = 12000
if len(inter) < MIN_INTERSECTION:
    raise ValueError(f"gene intersection {len(inter)} < {MIN_INTERSECTION}; "
                     "studies may use inconsistent gene-symbol conventions")

from src.data_loaders import NICHE_CRITICAL_GENES
niche_out = [g for g in NICHE_CRITICAL_GENES if g not in inter]
print(f"niche genes in intersection: {len(NICHE_CRITICAL_GENES) - len(niche_out)}"
      f"/{len(NICHE_CRITICAL_GENES)}")
if niche_out:
    print(f"WARNING: niche genes LOST to inner join (not in all 3 studies): {niche_out}")

adata = ad.concat(list(adatas.values()), join="inner", label="study_id",
                  keys=list(adatas.keys()), index_unique="-", merge="same")
adata.obs["study_id"] = adata.obs["study_id"].astype("category")
del adatas; gc.collect()

print("\nconcatenated:", adata.shape)
print(adata.obs["study_id"].value_counts())
_ram("after concat")

## 4 — Feature selection

### 4a — Highly variable genes (`seurat_v3`, batch-aware) + niche-gene injection

`seurat_v3` ranks genes by variance-stabilized dispersion on **raw counts**; with
`batch_key="study_id"` it ranks within each study and combines by median rank, so HVGs are not
dominated by the largest study. The scVI gene set is the 3000 HVGs **plus** any niche-critical
gene not already selected (force-injected so APOE/TREM2/GFAP/etc. are always in the latent space).
The full-gene object is kept; `in_scvi` marks the training feature set.

In [ ]:
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=3000,
                            batch_key="study_id", subset=False)

hvg = adata.var_names[adata.var["highly_variable"]].tolist()
inject = [g for g in NICHE_CRITICAL_GENES if g in adata.var_names and g not in hvg]
scvi_genes = list(dict.fromkeys(hvg + inject))
adata.var["in_scvi"] = adata.var_names.isin(scvi_genes)

print(f"HVG (seurat_v3, batch-aware): {len(hvg)}")
print(f"niche genes already HVG:      {[g for g in NICHE_CRITICAL_GENES if g in hvg]}")
print(f"niche genes injected (not HVG): {inject}")
print(f"scVI gene set: {int(adata.var['in_scvi'].sum())}")
_ram("after HVG")

## 5 — scVI integration

### 5a — Setup + train scVI

`setup_anndata` with `layer=None` uses `.X` (raw counts) and `batch_key="study_id"`. Model
choices (all non-default, justified in the title cell): `n_latent=30`, `n_layers=2`,
`gene_likelihood="nb"`. Seed fixed at 0. `max_epochs=None` lets scVI's heuristic auto-scale
epochs down for the large cell count; early stopping caps it on the validation ELBO. **Fails loud
if no GPU** — training on CPU at this scale is impractical.

In [ ]:
import torch, scvi
import matplotlib.pyplot as plt

scvi.settings.seed = 0

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. scVI at ~763k cells needs a GPU runtime (A100). "
                       "Switch runtime type to GPU and re-run from 1a.")
print("GPU:", torch.cuda.get_device_name(0))

adata_hvg = adata[:, adata.var["in_scvi"]].copy()    # ~763k x ~3008, raw counts in .X
scvi.model.SCVI.setup_anndata(adata_hvg, batch_key="study_id")

model = scvi.model.SCVI(
    adata_hvg,
    n_latent=30,            # default 10
    n_layers=2,             # default 1
    n_hidden=128,
    gene_likelihood="nb",   # default "zinb"
    dropout_rate=0.1,
)
model.train(
    max_epochs=150,             # explicit: heuristic gives ~10 at 763k (still descending at ep 9)
    early_stopping=True,
    early_stopping_patience=10,
    check_val_every_n_epoch=1,
)

hist = model.history
fig, ax = plt.subplots(figsize=(6, 4))
hist["elbo_train"].plot(ax=ax, label="train")
if "elbo_validation" in hist:
    hist["elbo_validation"].plot(ax=ax, label="validation")
ax.set_xlabel("epoch"); ax.set_ylabel("ELBO"); ax.legend()
ax.set_title("scVI training")
plt.show()
print(f"epochs trained: {len(hist['elbo_train'])}")
_ram("after scVI train")

### 5b — Latent representation + neighbors + Leiden + UMAP

Write the 30-d scVI latent onto the **full-gene** object as `obsm["X_scVI"]`, build the
neighbor graph on it, cluster (Leiden, `key="leiden_scvi"`), and embed with UMAP. All downstream
notebooks reload this object and use `X_scVI` as the integrated representation.

In [ ]:
adata.obsm["X_scVI"] = model.get_latent_representation(adata_hvg)
print("latent:", adata.obsm["X_scVI"].shape)
del adata_hvg; gc.collect()

sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
sc.tl.leiden(adata, resolution=1.0, key_added="leiden_scvi",
             flavor="igraph", n_iterations=2, directed=False)
sc.tl.umap(adata, min_dist=0.3)
print("Leiden clusters:", adata.obs["leiden_scvi"].nunique())
_ram("after umap")

## 6 — Integration diagnostics (label-free)

### 6a — UMAP overlays: study / diagnosis / APOE / clusters + glia markers

Qualitative read of the integration. A grid plus **per-panel full-size** plots (study mixing is
hard to judge from a small grid). Then per-marker UMAPs for canonical astro / microglia / context
markers, computed on a temporary log-normalized layer (the object holds raw counts; normalization
here is for visualization only and is discarded). Glia should form coherent, study-mixed regions —
if astro/micro markers light up tight, well-mixed territories, ITS subsetting in colab_05 is on
solid ground.

In [ ]:
# grid overview
sc.pl.umap(adata, color=["study_id", "diagnosis", "apoe_carrier", "leiden_scvi"],
           ncols=2, wspace=0.3, show=True)

# per-panel full-size for the categoricals (mixing is hard to judge in a small grid)
for c in ["study_id", "diagnosis", "apoe_carrier", "leiden_scvi"]:
    sc.pl.umap(adata, color=c, show=True, title=c,
               legend_loc="on data" if c == "leiden_scvi" else "right margin")

# canonical markers (only those present in the intersection)
markers = [g for g in ["AQP4", "GFAP", "SLC1A2",                 # astrocyte
                       "CSF1R", "AIF1", "TREM2", "P2RY12",       # microglia
                       "RBFOX3", "SNAP25", "MBP", "MOBP"]         # neuron / oligo context
           if g in adata.var_names]
print("markers shown:", markers)

adata.layers["lognorm_vis"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4, layer="lognorm_vis")
sc.pp.log1p(adata, layer="lognorm_vis")
for g in markers:
    sc.pl.umap(adata, color=g, layer="lognorm_vis", cmap="viridis", show=True, title=g)
del adata.layers["lognorm_vis"]
adata.uns.pop("log1p", None)   # sc.pp.log1p writes this key; remove so colab_05 isn't
gc.collect()                     # silently skipped if it calls log1p on raw .X

### 6b — Label-free residual-batch screen (per-cluster study composition)

For each Leiden cluster, the fraction of cells from each study. A cluster drawn almost entirely
from one study is a candidate residual-batch artifact (or a genuine study-private population —
distinguished only after annotation). This is **not** scIB: bio-conservation metrics need
cell-type labels and are deferred to colab_06. This table is the colab_04 integration health
check.

In [ ]:
comp = pd.crosstab(adata.obs["leiden_scvi"], adata.obs["study_id"], normalize="index")
print("per-cluster study composition (row-normalized):")
with pd.option_context("display.max_rows", None, "display.width", 120):
    print(comp.round(3))

DOMINANCE = 0.90
dominated = comp.index[comp.max(axis=1) >= DOMINANCE].tolist()
print(f"\nclusters >= {DOMINANCE:.0%} one study (possible residual batch OR study-private): "
      f"{dominated}")
print("\noverall study fractions:",
      adata.obs["study_id"].value_counts(normalize=True).round(3).to_dict())
print("\nNOTE: full scIB (iLISI/kBET batch + NMI/ARI/ASW bio-conservation) is DEFERRED to "
      "colab_06 — bio-conservation needs cell-type labels (annotation = colab_05).")

## 7 — Save integrated object + scVI model

### 7a — Save integrated AnnData (full-gene counts + `X_scVI`) to Drive

Write the full-gene object with raw counts in `.X` (CSR), the scVI latent in `obsm["X_scVI"]`,
UMAP, Leiden labels, and the harmonized obs. This is the single hand-off object for colab_05
(annotation + subset) and colab_06 (2nd method + scIB). Drive-only (gitignored).

In [ ]:
INTEGRATED_DIR = os.path.join(DRIVE_ROOT, "integrated")
os.makedirs(INTEGRATED_DIR, exist_ok=True)

if not sp.issparse(adata.X) or adata.X.getformat() != "csr":
    adata.X = sp.csr_matrix(adata.X)

INTEGRATED_PATH = os.path.join(INTEGRATED_DIR, "scvi_integrated_full.h5ad")
adata.write_h5ad(INTEGRATED_PATH, compression="gzip")
print(f"Wrote {INTEGRATED_PATH}  ({os.path.getsize(INTEGRATED_PATH)/1e9:.2f} GB)  shape={adata.shape}")
print("  .obsm:", list(adata.obsm.keys()))
print("  .obs :", list(adata.obs.columns))

### 7b — Save trained scVI model

Save the model directory so colab_06 can initialize **scANVI** from this scVI model
(`scANVI.from_scvi_model`) instead of retraining from scratch. `save_anndata=False` — the data is
already on Drive via 7a.

In [ ]:
MODEL_DIR = os.path.join(DRIVE_ROOT, "models", "scvi_colab04")
os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)
model.save(MODEL_DIR, overwrite=True, save_anndata=False)
print("scVI model saved to", MODEL_DIR)
print("  (colab_06 scANVI can init from this via scvi.model.SCANVI.from_scvi_model)")

## 8 — Handoff to annotation

### 8a — Write integration trace to audit report + summary + commit instructions

Append an `integration_scvi` entry to the cumulative `outputs/audit_report.json` (cells, gene
intersection, HVG count, clusters, study-dominated clusters), then print the artifact list and the
WSL commit commands. Committable: env freeze/JSON + `audit_report.json`. Drive-only: the integrated
h5ad and the scVI model.

In [ ]:
import shlex

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

report["integration_scvi"] = {
    "status": "computed",
    "date": TODAY,
    "n_cells": int(adata.n_obs),
    "n_genes_intersection": int(adata.n_vars),
    "n_hvg": int((adata.var["highly_variable"]).sum()),
    "n_scvi_genes": int(adata.var["in_scvi"].sum()),
    "batch_key": "study_id",
    "n_latent": 30,
    "gene_likelihood": "nb",
    "n_leiden_clusters": int(adata.obs["leiden_scvi"].nunique()),
    "study_dominated_clusters": (
        pd.crosstab(adata.obs["leiden_scvi"], adata.obs["study_id"], normalize="index")
        .pipe(lambda _c: _c.index[_c.max(axis=1) >= 0.90].tolist())
    ),
    "study_fractions": adata.obs["study_id"].value_counts(normalize=True).round(4).to_dict(),
}
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print("=== Artifacts ===")
print("  committable: ", FREEZE_PATH)
print("  committable: ", ENV_JSON_PATH)
print("  committable: ", AUDIT_REPORT_PATH)
print("  drive-only : ", INTEGRATED_PATH)
print("  drive-only : ", MODEL_DIR)

rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH)]
print("\n=== Commit + push (from WSL — Colab has no git creds) ===")
print("  cd /content/ad-glia-fm-prep && git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /content/ad-glia-fm-prep && git commit -m "
      "'colab_04: scVI integration run record (env + audit trace)'")
print("  cd /content/ad-glia-fm-prep && git push")

### Carried forward to colab_05 / colab_06

**colab_05 (annotation + subset):**
- Annotate the integrated object: Leiden + canonical markers; use `orig_celltype` (SEA-AD only)
  as a cross-check, not ground truth.
- Subset astrocytes + microglia (ITS) from the **full-gene** object → save the glia subset for the
  FM notebooks.
- **Carried-forward niche-mito diagnostic (still open from colab_01/03):** among annotated
  astrocytes, check `pct_counts_mt` against the 5% ceiling for pile-up — the genuinely clean
  per-cell-type check that Li (no labels) and Haney (no labels) could not run at QC time.

**colab_06 (2nd method + scIB):**
- Second integration method — **decision pending**: **scANVI** (label-aware, inits from the saved
  scVI model, needs colab_05 labels; the locked plan favors it for the APOE×study confound) vs
  **Harmony** (label-free, runs independently). Run full **scIB** (batch: iLISI/kBET/graph
  connectivity; bio: NMI/ARI/ASW vs cell type) comparing scVI vs the 2nd method, and articulate
  tradeoffs.
- **Audit B flip:** SEA-AD and Li entries are still `skipped` in `audit_report.json` — flip to
  `pass` in the colab_06 audit cell once confirmed.